<a href="https://colab.research.google.com/github/Mohammed-Abdul-Rafe-Sajid/Deep-Learning-/blob/main/LAB_10_Encoder_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Complete Seq2Seq Pipeline

In [ ]:
import numpy as np
from keras.models import Model
from keras.layers import Input, LSTM, Dense

In [ ]:
# Dummy vocabulary sizes
num_encoder_tokens = 8
num_decoder_tokens = 8
latent_dim = 64

In [ ]:
# Dummy training data
encoder_input_data = np.random.random((1,5,num_encoder_tokens))
decoder_input_data = np.random.random((1,5,num_decoder_tokens))
decoder_target_data = np.random.random((1,5,num_decoder_tokens))

In [ ]:
# Encoder (variable length sequence)
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder(encoder_inputs)
encoder_states = [state_h, state_c]

In [ ]:
# Decoder
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs,
                                     initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:

# Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy")

In [ ]:

# Train
model.fit([encoder_input_data, decoder_input_data],
          decoder_target_data, epochs=5)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 39.1030
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 39.0868
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - loss: 39.0770
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 39.0752
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 39.0841


In [ ]:
import numpy as np
# Simulated translation
pred = model.predict([encoder_input_data, decoder_input_data])
print("Prediction output:")
print(pred)
translated_word = np.argmax(pred[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
Prediction output:
[[[0.05409698 0.03397463 0.05015925 0.03018797 0.04292605 0.05103327
   0.03690953 0.04489656 0.04161699 0.03292911 0.04575212 0.0543679
   0.04397814 0.03845226 0.03795953 0.04822593 0.03993405 0.03490456
   0.03357361 0.0325489  0.04816239 0.03734494 0.03743175 0.04863364]
  [0.05555505 0.03217316 0.04639852 0.03044461 0.04444814 0.05241928
   0.03858341 0.04299988 0.03906642 0.03111735 0.04628645 0.05968696
   0.04567578 0.03931381 0.03746386 0.0474032  0.04028032 0.03551441
   0.03110677 0.02987257 0.04756394 0.03735542 0.03848311 0.05078763]
  [0.05471012 0.03278701 0.0452351  0.03039552 0.04660929 0.05344867
   0.04002439 0.0412897  0.0390672  0.03117542 0.04654618 0.0635222
   0.04694819 0.03783305 0.03449463 0.04951411 0.0415266  0.03502988
   0.0304861  0.02741513 0.04866198 0.03639637 0.03924764 0.04763554]
  [0.05348293 0.03306839 0.04519889 0.03024256 0.04775863 0.05219627
   0.04193143 0.04114888 0.03628621 0.031055

This index corresponds to a word in the target vocabulary dictionary.
| Index | Word    |
| ----- | ------- |
| 0     | salut   |
| 1     | merci   |
| 2     | bonjour |


In [ ]:
print("Input sentence: hello")
print("Translated token index:", translated_word)
print("Translated sentence: bonjour (example)")

Input sentence: hello
Translated token index: 11
Translated sentence: bonjour (example)


In [ ]:
import numpy as np
from keras.models import Model
from keras.layers import Input, LSTM, Dense

# --- Configuration Parameters ---
# num_encoder_tokens: The size of the input vocabulary for the encoder. Each token will be one-hot encoded.
num_encoder_tokens = 16
# num_decoder_tokens: The size of the output vocabulary for the decoder. Each token will be one-hot encoded.
num_decoder_tokens = 24
# latent_dim: The dimensionality of the latent space (the hidden state size of the LSTM units).
# This determines the capacity of the model to learn complex patterns.
latent_dim = 128

# --- Dummy Training Data ---
# encoder_input_data: Input sequences for the encoder.
# Shape: (number_of_samples, max_input_sequence_length, num_encoder_tokens)
# For example, (1, 5, num_encoder_tokens) means 1 sample, sequence length of 5, with each token being a one-hot vector of size num_encoder_tokens.
encoder_input_data = np.random.random((1,5,num_encoder_tokens))

# decoder_input_data: Input sequences for the decoder (shifted version of target sequence).
# Shape: (number_of_samples, max_output_sequence_length, num_decoder_tokens)
# Used for teacher forcing during training.
decoder_input_data = np.random.random((1,5,num_decoder_tokens))

# decoder_target_data: Target sequences for the decoder (actual output sequence).
# Shape: (number_of_samples, max_output_sequence_length, num_decoder_tokens)
# Used to calculate the loss during training. This is typically one-hot encoded.
decoder_target_data = np.random.random((1,5,num_decoder_tokens))

# --- Encoder Definition ---
# encoder_inputs: Input layer for the encoder. Defines the shape of the input sequence.
# shape=(None, num_encoder_tokens) means it can accept sequences of any length (None) with each element being a vector of size num_encoder_tokens.
encoder_inputs = Input(shape=(None, num_encoder_tokens))

# encoder: LSTM layer for the encoder.
# latent_dim: Number of LSTM units.
# return_state=True: Ensures the LSTM returns its final hidden state (state_h) and cell state (state_c).
# These states capture the "context" of the input sequence and will be passed to the decoder.
encoder = LSTM(latent_dim, return_state=True)

# _: Discards the output sequence of the encoder (we only need the states).
# state_h: The final hidden state of the encoder LSTM.
# state_c: The final cell state of the encoder LSTM.
_, state_h, state_c = encoder(encoder_inputs)

# encoder_states: A list containing the final hidden and cell states of the encoder.
# These will serve as the initial states for the decoder.
encoder_states = [state_h, state_c]

# --- Decoder Definition ---
# decoder_inputs: Input layer for the decoder. Defines the shape of the target sequence (during training).
# shape=(None, num_decoder_tokens) means it can accept sequences of any length (None) with each element being a vector of size num_decoder_tokens.
decoder_inputs = Input(shape=(None, num_decoder_tokens))

# decoder_lstm: LSTM layer for the decoder.
# latent_dim: Number of LSTM units, same as the encoder to match state dimensions.
# return_sequences=True: Ensures the LSTM returns the full sequence of outputs, not just the last one.
# return_state=True: (Optional for training, but useful for inference) Returns the final states of the decoder.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)

# decoder_outputs: The output sequence of the decoder LSTM.
# initial_state=encoder_states: The decoder is initialized with the context states from the encoder.
# _: Discards the final hidden and cell states of the decoder LSTM for this training setup.
decoder_outputs, _, _ = decoder_lstm(decoder_inputs,
                                     initial_state=encoder_states)

# decoder_dense: Dense layer applied to the decoder's output sequence.
# num_decoder_tokens: Output dimension matches the decoder's vocabulary size.
# activation="softmax": Softmax activation to produce a probability distribution over the output vocabulary for each time step.
decoder_dense = Dense(num_decoder_tokens, activation="softmax")

# decoder_outputs: The final output of the decoder, after passing through the dense layer.
decoder_outputs = decoder_dense(decoder_outputs)

# --- Model Compilation ---
# model: The complete Seq2Seq training model.
# inputs: A list of the encoder input and decoder input layers.
# outputs: The final decoder output layer.
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# optimizer="adam": Optimization algorithm used for training the model.
# loss="categorical_crossentropy": Loss function suitable for multi-class classification where targets are one-hot encoded.
model.compile(optimizer="adam", loss="categorical_crossentropy")

# --- Model Training ---
# epochs: The number of times the model will iterate over the entire training dataset.
model.fit([encoder_input_data, decoder_input_data],
          decoder_target_data, epochs=5)

# --- Model Prediction (Simulated Translation) ---
# pred: The predicted output probabilities from the model for the given input.
# The input data is the same as training data for this dummy example.
pred = model.predict([encoder_input_data, decoder_input_data])

print("Prediction output:")
print(pred)

# translated_word: The index of the most probable word (token) from the first time step of the first sample.
# np.argmax: Returns the index of the maximum value along a given axis.
# pred[0][0]: Accesses the first sample and the first time step's output probabilities.
translated_word = np.argmax(pred[0][0])

# --- Dictionary for reference ---
# This table shows example mappings from index to word in the target vocabulary.
# | Index | Word    |
# | ----- | ------- |
# | 0     | salut   |
# | 1     | merci   |
# | 2     | bonjour |

print("Input sentence: hello")
print("Translated token index:", translated_word)
print("Translated sentence: bonjour (example)")


Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 39.3848
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 39.3736
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 39.3709
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 39.3751
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 39.3863
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
Prediction output:
[[[0.04567776 0.03876467 0.05873771 0.03608679 0.03571979 0.04283049
   0.03008082 0.04087757 0.04608854 0.04656087 0.03303232 0.04004216
   0.03977509 0.043499   0.04158188 0.03302608 0.04513836 0.05035086
   0.04190613 0.04253711 0.04781784 0.0358974  0.0422769  0.04169379]
  [0.04907591 0.04181502 0.05573199 0.03290765 0.03559101 0.0448089
   0.02885918 0.04281569 0.04492241 0.04454919 0.03245434 0.04102722
   0.03751486 0.04555485 0.0423424  0.03439127 0.04277462 0.05003852
   0.03776145 0.04162412 0.04960063 0.03843377 0.03964145 0.04576356]
  [0.05232848 0.04397303 0.05423074 0.02945896 0.03408121 0.04581646
  

In [2]:
!pip uninstall -y transformers tokenizers
!pip install transformers==4.41.2 tensorflow

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 59.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1


In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
# !pip uninstall -y transformers tokenizers
# !pip install transformers==4.41.2 tensorflow
# import os
# os.kill(os.getpid(), 9)


import tensorflow as tf
import numpy as np
from transformers import BertTokenizer, TFBertForSequenceClassification

#Load Pretrained BERT Model and Tokenizer
#12 transformer layers  #bert-base-uncased
#110 million parameters
#input text is converted into correct token IDs.
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Create Sample Dataset
texts = ["I love this movie", "This film is terrible"]
labels = [1, 0]   # 1 = Positive, 0 = Negative

# Tokenizer converts text into BERT inputs
# converts labels to TensorFlow format
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="tf")
labels = tf.constant(labels)

#  from_logits=True BERT outputs raw scores (logits)
#The loss function internally applies softmax.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

#inputs.data: Contains tokenized input tensors
#labels: correct sentiment values
model.fit(inputs.data, labels, epochs=2)

# Test the Model
test_text = ["This movie is terrible"]
test_input = tokenizer(test_text, return_tensors="tf")

prediction = model.predict(test_input.data)
sentiment = np.argmax(prediction.logits)

print("Sentence:", test_text[0])

if sentiment == 1:
    print("Sentiment: Positive")
else:
    print("Sentiment: Negative")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Epoch 1/2
1/1 [==============================] - 59s 59s/step - loss: 1.0207 - accuracy: 0.5000
Epoch 2/2
1/1 [==============================] - 4s 4s/step
Sentence: This movie is terrible
Sentiment: Negative
